# Crypto Exchange API Tester

Manual notebook for testing the project Bybit and OKX API integration paths.

Run this notebook from the `backend` directory with the backend virtual environment kernel. It uses the existing project clients and normalizers:

- `core.crypto_exchange_clients.BybitClient`
- `core.crypto_exchange_clients.OKXClient`
- `services.crypto_exchange.normalize_bybit_spot_execution`
- `services.crypto_exchange.normalize_okx_spot_fill`
- optional `services.broker_api.BybitAPI` / `OKXAPI` using credentials stored in the app database

The notebook does not print API secrets and does not write transactions to the database unless `ALLOW_DB_WRITES = True` is set explicitly.

## Credential Setup

Preferred path: use credentials stored through the app UI.

1. Create a `Broker` for Bybit or OKX.
2. Create a broker account under that broker.
3. Save the exchange API credentials through `User Settings -> Broker API credentials`.
4. Run the `Database Credential Discovery` cells below.

This notebook defaults to `user_id=1`, because the local test database has the ByBit and OKX broker/account/token rows under that user. You can override that with environment variables if needed:

- `PM_USER_ID=1`
- optional `PM_ACCOUNT_ID=<account id>`
- optional `CRYPTO_API_PROVIDER=auto|bybit|okx`
- optional `CRYPTO_API_LOOKBACK_DAYS=7`

Direct environment credentials are still supported as a fallback for the direct client smoke tests, but they are not required when using stored app credentials.

Bybit key guidance:

- Create either a mainnet key or a testnet key. Testnet keys are separate from mainnet keys.
- Required values: API key and API secret.
- Required read permissions: account/wallet read, transaction log read, order/execution history read for the categories you want to test.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- Environment fallback variables:
  - `BYBIT_API_KEY`
  - `BYBIT_API_SECRET`
  - `BYBIT_TESTNET=1` for testnet, otherwise `0`
  - optional `BYBIT_ACCOUNT_TYPE=UNIFIED`
  - optional `BYBIT_CATEGORY=spot`

OKX key guidance:

- Create a read-only API key. OKX also requires the passphrase created with the key.
- Required values: API key, API secret, passphrase.
- Required read permissions: account read and trade/fills history read. For rewards/transfers testing, also allow read access to funding/account bills.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- For demo/simulated testing, create or use demo trading credentials and set `OKX_SIMULATED_TRADING=1`.
- Environment fallback variables:
  - `OKX_API_KEY`
  - `OKX_API_SECRET`
  - `OKX_PASSPHRASE`
  - `OKX_SIMULATED_TRADING=1` for demo/simulated trading, otherwise `0`


In [5]:
import os
import sys
from datetime import datetime, timedelta, timezone
from pprint import pprint

# Keep the backend directory importable when the notebook is opened from here.
backend_dir = os.path.abspath('.')
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

from notebook_setup import setup_django

setup_django()

from common.models import Accounts, Transactions
from services.broker_api import BybitAPI, OKXAPI
from core.crypto_exchange_clients import BybitClient, CryptoExchangeAPIError, OKXClient
from services.crypto_exchange import (
    normalize_bybit_spot_execution,
    normalize_okx_spot_fill,
    persist_crypto_exchange_event,
)
from users.models import BybitApiToken, CustomUser, OKXApiToken


Django was already initialized!


In [6]:
# Smoke check: verify every module this notebook depends on still imports.
# Self-contained — run it any time after a refactor that touches services/ or core/.
# Catches moved/renamed modules BEFORE you hit them mid-notebook. Safe to run alone.
import importlib
from notebook_setup import setup_django
setup_django()  # idempotent

_checks = {
    "services.broker_api": ["BybitAPI", "OKXAPI", "get_broker_api"],
    "core.crypto_exchange_clients": ["BybitClient", "OKXClient", "CryptoExchangeAPIError"],
    "services.crypto_exchange": [
        "normalize_bybit_spot_execution",
        "normalize_okx_spot_fill",
        "persist_crypto_exchange_event",
    ],
    "common.models": ["Accounts", "Transactions"],
    "users.models": ["BybitApiToken", "CustomUser", "OKXApiToken"],
}

_failures = []
for module_path, names in _checks.items():
    try:
        mod = importlib.import_module(module_path)
    except Exception as exc:
        _failures.append(f"{module_path}: import failed -> {exc!r}")
        continue
    for name in names:
        if not hasattr(mod, name):
            _failures.append(f"{module_path}: missing attribute {name!r}")

if _failures:
    raise AssertionError("; ".join(_failures))
print("smoke OK —", len(_checks), "modules,", sum(len(v) for v in _checks.values()), "symbols verified")


Django was already initialized!
smoke OK — 5 modules, 14 symbols verified


In [7]:
def env_bool(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y'}


def masked(value):
    if not value:
        return '<missing>'
    if len(value) <= 8:
        return '<set>'
    return f'{value[:4]}...{value[-4:]}'


def require_env(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise RuntimeError(
            'Missing environment variables: ' + ', '.join(missing)
        )


def date_range_ms(days=7):
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)
    return str(int(start.timestamp() * 1000)), str(int(end.timestamp() * 1000))


def optional_int_env(name):
    value = os.getenv(name)
    if value in (None, ''):
        return None
    return int(value)


DEFAULT_USER_ID = int(os.getenv('PM_USER_ID', '1'))
SELECTED_ACCOUNT_ID = optional_int_env('PM_ACCOUNT_ID')
SELECTED_PROVIDER = os.getenv('CRYPTO_API_PROVIDER', 'auto').strip().lower()
LOOKBACK_DAYS = int(os.getenv('CRYPTO_API_LOOKBACK_DAYS', '7'))
TRANSACTION_PROVIDER = os.getenv('CRYPTO_API_TRANSACTION_PROVIDER', 'all').strip().lower()
MAX_TRANSACTION_PAGES = int(os.getenv('CRYPTO_API_MAX_TRANSACTION_PAGES', '20'))
ALLOW_DB_WRITES = env_bool('CRYPTO_API_TEST_ALLOW_DB_WRITES', False)

print('Runtime config:')
print('  DEFAULT_USER_ID:', DEFAULT_USER_ID)
print('  SELECTED_PROVIDER:', SELECTED_PROVIDER)
print('  SELECTED_ACCOUNT_ID:', SELECTED_ACCOUNT_ID or '<auto>')
print('  LOOKBACK_DAYS:', LOOKBACK_DAYS)
print('  TRANSACTION_PROVIDER:', TRANSACTION_PROVIDER)
print('  MAX_TRANSACTION_PAGES:', MAX_TRANSACTION_PAGES)
print('  ALLOW_DB_WRITES:', ALLOW_DB_WRITES)
print('Environment fallback credential preview:')
print('  BYBIT_API_KEY:', masked(os.getenv('BYBIT_API_KEY')))
print('  OKX_API_KEY:', masked(os.getenv('OKX_API_KEY')))


Runtime config:
  DEFAULT_USER_ID: 1
  SELECTED_PROVIDER: auto
  SELECTED_ACCOUNT_ID: <auto>
  LOOKBACK_DAYS: 7
  TRANSACTION_PROVIDER: all
  MAX_TRANSACTION_PAGES: 20
  ALLOW_DB_WRITES: False
Environment fallback credential preview:
  BYBIT_API_KEY: <missing>
  OKX_API_KEY: <missing>


## Database Credential Discovery

This is the preferred path when credentials were saved through the app UI. It finds active Bybit/OKX tokens for `DEFAULT_USER_ID`, finds broker accounts under the token brokers, and chooses a provider/account for adapter tests.

Current local defaults are expected to work with `user_id=1`:

- ByBit broker/account/token under user 1
- OKX broker/account/token under user 1

Set `CRYPTO_API_PROVIDER=bybit` or `CRYPTO_API_PROVIDER=okx` to force one provider. Set `PM_ACCOUNT_ID` only when you want to override the automatically selected account.


In [8]:
def account_for_broker(broker, preferred_id=None):
    accounts = Accounts.objects.select_related('broker').filter(broker=broker).order_by('id')
    if preferred_id is not None:
        preferred = accounts.filter(id=preferred_id).first()
        if preferred is not None:
            return preferred
    return accounts.first()


def print_token_summary(label, tokens):
    print(f'{label}: {len(tokens)} active token(s)')
    for token in tokens:
        account = account_for_broker(token.broker, SELECTED_ACCOUNT_ID)
        print(
            '  ',
            {
                'token_id': token.id,
                'broker_id': token.broker_id,
                'broker': token.broker.name,
                'api_key': masked(token.api_key),
                'account_id': account.id if account else None,
                'account': account.name if account else None,
                'testnet': getattr(token, 'testnet', None),
                'simulated_trading': getattr(token, 'simulated_trading', None),
            },
        )


if SELECTED_PROVIDER not in {'auto', 'bybit', 'okx'}:
    raise RuntimeError('CRYPTO_API_PROVIDER must be auto, bybit, or okx')

user = CustomUser.objects.get(id=DEFAULT_USER_ID)
bybit_tokens = list(
    BybitApiToken.objects.select_related('broker')
    .filter(user=user, is_active=True)
    .order_by('id')
)
okx_tokens = list(
    OKXApiToken.objects.select_related('broker')
    .filter(user=user, is_active=True)
    .order_by('id')
)

print('Selected user:', {'id': user.id, 'username': user.username, 'email': user.email})
print_token_summary('Bybit', bybit_tokens)
print_token_summary('OKX', okx_tokens)

bybit_db_token = bybit_tokens[0] if bybit_tokens else None
okx_db_token = okx_tokens[0] if okx_tokens else None
bybit_db_account = account_for_broker(bybit_db_token.broker, SELECTED_ACCOUNT_ID) if bybit_db_token else None
okx_db_account = account_for_broker(okx_db_token.broker, SELECTED_ACCOUNT_ID) if okx_db_token else None

if SELECTED_PROVIDER == 'bybit':
    selected_provider = 'bybit'
    selected_token = bybit_db_token
    selected_account = bybit_db_account
elif SELECTED_PROVIDER == 'okx':
    selected_provider = 'okx'
    selected_token = okx_db_token
    selected_account = okx_db_account
elif bybit_db_token:
    selected_provider = 'bybit'
    selected_token = bybit_db_token
    selected_account = bybit_db_account
elif okx_db_token:
    selected_provider = 'okx'
    selected_token = okx_db_token
    selected_account = okx_db_account
else:
    selected_provider = None
    selected_token = None
    selected_account = None

if selected_token is None or selected_account is None:
    raise RuntimeError(
        'No usable stored exchange token/account was found. Save credentials through User Settings and create a broker account first.'
    )

print('Selected adapter target:')
print(
    {
        'provider': selected_provider,
        'token_id': selected_token.id,
        'broker_id': selected_token.broker_id,
        'broker': selected_token.broker.name,
        'account_id': selected_account.id,
        'account': selected_account.name,
        'account_native_id': selected_account.native_id,
    }
)



def get_bybit_client_from_db_or_env():
    if bybit_db_token is not None:
        return BybitClient(
            api_key=bybit_db_token.api_key,
            api_secret=bybit_db_token.get_api_secret(user),
            testnet=bybit_db_token.testnet,
        )
    require_env('BYBIT_API_KEY', 'BYBIT_API_SECRET')
    return BybitClient(
        api_key=os.environ['BYBIT_API_KEY'],
        api_secret=os.environ['BYBIT_API_SECRET'],
        testnet=env_bool('BYBIT_TESTNET', False),
    )


def get_okx_client_from_db_or_env():
    if okx_db_token is not None:
        return OKXClient(
            api_key=okx_db_token.api_key,
            api_secret=okx_db_token.get_api_secret(user),
            passphrase=okx_db_token.get_passphrase(user),
            simulated_trading=okx_db_token.simulated_trading,
        )
    require_env('OKX_API_KEY', 'OKX_API_SECRET', 'OKX_PASSPHRASE')
    return OKXClient(
        api_key=os.environ['OKX_API_KEY'],
        api_secret=os.environ['OKX_API_SECRET'],
        passphrase=os.environ['OKX_PASSPHRASE'],
        simulated_trading=env_bool('OKX_SIMULATED_TRADING', False),
    )


Selected user: {'id': 1, 'username': 'Y', 'email': 'test@test.com'}
Bybit: 1 active token(s)
   {'token_id': 1, 'broker_id': 29, 'broker': 'ByBit', 'api_key': 'BpZB...vlkr', 'account_id': 17, 'account': 'Main', 'testnet': False, 'simulated_trading': None}
OKX: 1 active token(s)
   {'token_id': 1, 'broker_id': 30, 'broker': 'OKX', 'api_key': 'fc82...373d', 'account_id': 18, 'account': 'Main', 'testnet': None, 'simulated_trading': False}
Selected adapter target:
{'provider': 'bybit', 'token_id': 1, 'broker_id': 29, 'broker': 'ByBit', 'account_id': 17, 'account': 'Main', 'account_native_id': None}


## Local App Account Inventory

Lists the portfolio app's broker/account rows for the selected user, with active crypto token metadata. This answers the local "which broker accounts are configured?" question before calling exchange APIs.


In [9]:
app_accounts = []
for account_row in Accounts.objects.select_related('broker').filter(
    broker__investor=user
).order_by('broker__name', 'id'):
    bybit_token_count = account_row.broker.bybit_tokens.filter(
        user=user, is_active=True
    ).count()
    okx_token_count = account_row.broker.okx_tokens.filter(
        user=user, is_active=True
    ).count()
    app_accounts.append(
        {
            'account_id': account_row.id,
            'account_name': account_row.name,
            'native_id': account_row.native_id,
            'broker_id': account_row.broker_id,
            'broker_name': account_row.broker.name,
            'broker_country': account_row.broker.country,
            'restricted': account_row.restricted,
            'is_active': account_row.is_active,
            'active_bybit_tokens': bybit_token_count,
            'active_okx_tokens': okx_token_count,
        }
    )

print(f'Configured app accounts for user {user.id}: {len(app_accounts)}')
pprint(app_accounts)


Configured app accounts for user 1: 7
[{'account_id': 17,
  'account_name': 'Main',
  'active_bybit_tokens': 1,
  'active_okx_tokens': 0,
  'broker_country': 'Global',
  'broker_id': 29,
  'broker_name': 'ByBit',
  'is_active': True,
  'native_id': None,
  'restricted': False},
 {'account_id': 1,
  'account_name': 'Investment',
  'active_bybit_tokens': 0,
  'active_okx_tokens': 0,
  'broker_country': 'UK',
  'broker_id': 23,
  'broker_name': 'Charles Stanley',
  'is_active': True,
  'native_id': '',
  'restricted': False},
 {'account_id': 10,
  'account_name': 'Test broker',
  'active_bybit_tokens': 0,
  'active_okx_tokens': 0,
  'broker_country': 'UK',
  'broker_id': 23,
  'broker_name': 'Charles Stanley',
  'is_active': True,
  'native_id': 'legacy_13',
  'restricted': False},
 {'account_id': 2,
  'account_name': 'Main',
  'active_bybit_tokens': 0,
  'active_okx_tokens': 0,
  'broker_country': 'US',
  'broker_id': 20,
  'broker_name': 'Interactive Brokers',
  'is_active': True,
  'na

## Exchange Account Details

Lists exchange-side account and balance details using the stored credentials. This is read-only and does not persist anything.

Bybit is queried through wallet balance by account type. OKX is queried through trading account balance, funding balance, and account config where permissions allow it.


In [10]:
def is_nonzero(value):
    return str(value or '').strip() not in {'', '0', '0.0', '0.00', '0.00000000', '0.000000000000000000'}


def safe_private_get(label, client, path, params=None):
    try:
        return client.get_private(path, params or {})
    except CryptoExchangeAPIError as exc:
        print(f'{label} failed: {exc}')
        return None


def compact_bybit_wallet(data):
    rows = []
    for account_payload in data.get('result', {}).get('list', []):
        account_type = account_payload.get('accountType')
        for coin in account_payload.get('coin', []):
            if any(
                is_nonzero(coin.get(key))
                for key in ['walletBalance', 'equity', 'usdValue', 'locked', 'cumRealisedPnl']
            ):
                rows.append(
                    {
                        'provider': 'bybit',
                        'account_type': account_type,
                        'coin': coin.get('coin'),
                        'wallet_balance': coin.get('walletBalance'),
                        'equity': coin.get('equity'),
                        'usd_value': coin.get('usdValue'),
                        'locked': coin.get('locked'),
                        'cum_realised_pnl': coin.get('cumRealisedPnl'),
                    }
                )
    return rows


def compact_okx_trading_balance(data):
    rows = []
    for account_payload in data.get('data', []):
        for detail in account_payload.get('details', []):
            if any(
                is_nonzero(detail.get(key))
                for key in ['cashBal', 'availBal', 'eq', 'eqUsd', 'frozenBal', 'rewardBal']
            ):
                rows.append(
                    {
                        'provider': 'okx',
                        'account_type': 'trading',
                        'currency': detail.get('ccy'),
                        'cash_balance': detail.get('cashBal'),
                        'available_balance': detail.get('availBal'),
                        'equity': detail.get('eq'),
                        'equity_usd': detail.get('eqUsd'),
                        'frozen_balance': detail.get('frozenBal'),
                        'reward_balance': detail.get('rewardBal'),
                        'total_pnl': detail.get('totalPnl'),
                    }
                )
    return rows


def compact_okx_funding_balance(data):
    rows = []
    for detail in data.get('data', []):
        if any(is_nonzero(detail.get(key)) for key in ['bal', 'availBal', 'frozenBal']):
            rows.append(
                {
                    'provider': 'okx',
                    'account_type': 'funding',
                    'currency': detail.get('ccy'),
                    'balance': detail.get('bal'),
                    'available_balance': detail.get('availBal'),
                    'frozen_balance': detail.get('frozenBal'),
                }
            )
    return rows


exchange_account_details = []

if bybit_db_token is not None:
    bybit_for_accounts = get_bybit_client_from_db_or_env()
    for account_type in ['UNIFIED', 'SPOT', 'CONTRACT', 'FUND']:
        data = safe_private_get(
            f'Bybit wallet-balance {account_type}',
            bybit_for_accounts,
            '/v5/account/wallet-balance',
            {'accountType': account_type},
        )
        if data:
            exchange_account_details.extend(compact_bybit_wallet(data))

if okx_db_token is not None:
    okx_for_accounts = get_okx_client_from_db_or_env()
    config = safe_private_get('OKX account config', okx_for_accounts, '/api/v5/account/config')
    if config:
        print('OKX account config:')
        pprint(config.get('data', []))
    trading = safe_private_get('OKX trading balance', okx_for_accounts, '/api/v5/account/balance')
    if trading:
        exchange_account_details.extend(compact_okx_trading_balance(trading))
    funding = safe_private_get('OKX funding balance', okx_for_accounts, '/api/v5/asset/balances')
    if funding:
        exchange_account_details.extend(compact_okx_funding_balance(funding))

print(f'Exchange account/balance rows: {len(exchange_account_details)}')
pprint(exchange_account_details)


2026-07-26T19:02:53.785713Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:54.508857Z [debug    ] https://api.bybit.com:443 "GET /v5/account/wallet-balance?accountType=UNIFIED HTTP/1.1" 200 1348 [urllib3.connectionpool]
2026-07-26T19:02:54.515697Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:54.905522Z [debug    ] https://api.bybit.com:443 "GET /v5/account/wallet-balance?accountType=SPOT HTTP/1.1" 200 111 [urllib3.connectionpool]


Bybit wallet-balance SPOT failed: Bybit API error: accountType only support UNIFIED.


2026-07-26T19:02:54.913354Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:55.349722Z [debug    ] https://api.bybit.com:443 "GET /v5/account/wallet-balance?accountType=CONTRACT HTTP/1.1" 200 111 [urllib3.connectionpool]


Bybit wallet-balance CONTRACT failed: Bybit API error: accountType only support UNIFIED.


2026-07-26T19:02:55.357483Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:55.821315Z [debug    ] https://api.bybit.com:443 "GET /v5/account/wallet-balance?accountType=FUND HTTP/1.1" 200 111 [urllib3.connectionpool]


Bybit wallet-balance FUND failed: Bybit API error: accountType only support UNIFIED.


2026-07-26T19:02:55.829047Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:02:56.510473Z [debug    ] https://www.okx.com:443 "GET /api/v5/account/config HTTP/1.1" 200 None [urllib3.connectionpool]


OKX account config:
[{'acctLv': '2',
  'acctStpMode': 'cancel_maker',
  'autoLoan': False,
  'ctIsoMode': 'automatic',
  'enableSpotBorrow': False,
  'feeType': '0',
  'greeksType': 'PA',
  'ip': '',
  'kycLv': '2',
  'label': 'PM test',
  'level': 'Lv1',
  'levelTmp': '',
  'liquidationGear': '-1',
  'mainUid': '652654290649420911',
  'mgnIsoMode': 'auto_transfers_ccy',
  'opAuth': '1',
  'perm': 'read_only',
  'posMode': 'long_short_mode',
  'roleType': '0',
  'settleCcy': 'USD',
  'settleCcyList': [],
  'spotBorrowAutoRepay': False,
  'spotOffsetType': '',
  'spotRoleType': '0',
  'spotTraderInsts': [],
  'stgyType': '0',
  'traderInsts': [],
  'type': '0',
  'uid': '652654290649420911'}]


2026-07-26T19:02:56.565311Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:02:57.036840Z [debug    ] https://www.okx.com:443 "GET /api/v5/account/balance HTTP/1.1" 200 None [urllib3.connectionpool]
2026-07-26T19:02:57.042963Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:02:57.550551Z [debug    ] https://www.okx.com:443 "GET /api/v5/asset/balances HTTP/1.1" 200 None [urllib3.connectionpool]


Exchange account/balance rows: 8
[{'account_type': 'UNIFIED',
  'coin': 'BTC',
  'cum_realised_pnl': '-0.00113313',
  'equity': '0',
  'locked': '0',
  'provider': 'bybit',
  'usd_value': '0.00032336',
  'wallet_balance': '0'},
 {'account_type': 'UNIFIED',
  'coin': 'USDT',
  'cum_realised_pnl': '67.87386975',
  'equity': '67.34837361',
  'locked': '0',
  'provider': 'bybit',
  'usd_value': '67.29274385',
  'wallet_balance': '67.34837361'},
 {'account_type': 'trading',
  'available_balance': '0.0000000018920997',
  'cash_balance': '0.0000000018920997',
  'currency': 'BTC',
  'equity': '0.0000000018920997',
  'equity_usd': '0.00012235584367',
  'frozen_balance': '0',
  'provider': 'okx',
  'reward_balance': '0',
  'total_pnl': ''},
 {'account_type': 'trading',
  'available_balance': '0.00006039864375',
  'cash_balance': '0.00006039864375',
  'currency': 'TRUMP',
  'equity': '0.00006039864375',
  'equity_usd': '0.0000957318503437',
  'frozen_balance': '0',
  'provider': 'okx',
  'reward_

## Bybit Direct Client Smoke Tests

These cells test signed private requests directly against Bybit. They use the active Bybit token stored in the app database when available, and fall back to `BYBIT_*` environment variables only when no DB token exists.


In [11]:
bybit = get_bybit_client_from_db_or_env()
if bybit_db_token is not None:
    print(
        'Using DB Bybit token:',
        {
            'token_id': bybit_db_token.id,
            'broker': bybit_db_token.broker.name,
            'account_id': bybit_db_account.id if bybit_db_account else None,
            'testnet': bybit_db_token.testnet,
        },
    )
else:
    print('Using BYBIT_* environment credentials')

bybit_account_type = os.getenv('BYBIT_ACCOUNT_TYPE', 'UNIFIED')
wallet = bybit.get_private(
    '/v5/account/wallet-balance',
    {'accountType': bybit_account_type},
)

print('Bybit wallet response keys:', wallet.keys())
pprint(wallet.get('result', {}))


Using DB Bybit token: {'token_id': 1, 'broker': 'ByBit', 'account_id': 17, 'testnet': False}


2026-07-26T19:02:57.693353Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:58.222640Z [debug    ] https://api.bybit.com:443 "GET /v5/account/wallet-balance?accountType=UNIFIED HTTP/1.1" 200 1348 [urllib3.connectionpool]


Bybit wallet response keys: dict_keys(['retCode', 'retMsg', 'result', 'retExtInfo', 'time'])
{'list': [{'accountIMRate': '0',
           'accountIMRateByMp': '0',
           'accountLTV': '0',
           'accountMMRate': '0',
           'accountMMRateByMp': '0',
           'accountType': 'UNIFIED',
           'coin': [{'accruedInterest': '0',
                     'availableToBorrow': '',
                     'availableToWithdraw': '',
                     'bonus': '0',
                     'borrowAmount': '0.000000000000000000',
                     'coin': 'BTC',
                     'colRes': '0',
                     'collateralSwitch': True,
                     'cumRealisedPnl': '-0.00113313',
                     'equity': '0',
                     'locked': '0',
                     'marginCollateral': True,
                     'spotBorrow': '0',
                     'spotHedgingQty': '0',
                     'totalOrderIM': '',
                     'totalPositionIM': '',
    

In [12]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
bybit_category = os.getenv('BYBIT_CATEGORY', 'spot')
bybit_execution_params = {
    'category': bybit_category,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_executions = []
try:
    bybit_executions = list(bybit.iter_executions(bybit_execution_params))
except CryptoExchangeAPIError as exc:
    print('Bybit execution fetch failed:', exc)

print(f'Fetched {len(bybit_executions)} Bybit execution rows')
if bybit_executions:
    pprint(bybit_executions[0])

2026-07-26T19:02:58.343345Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:58.770460Z [debug    ] https://api.bybit.com:443 "GET /v5/execution/list?category=spot&endTime=1785092578341&limit=100&startTime=1784487778341 HTTP/1.1" 200 123 [urllib3.connectionpool]


Fetched 0 Bybit execution rows


In [13]:
bybit_log_params = {
    'accountType': bybit_account_type,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_log_rows = []
try:
    bybit_log_rows = list(bybit.iter_transaction_log(bybit_log_params))
except CryptoExchangeAPIError as exc:
    print('Bybit transaction-log fetch failed:', exc)

print(f'Fetched {len(bybit_log_rows)} Bybit transaction log rows')
if bybit_log_rows:
    pprint(bybit_log_rows[0])

2026-07-26T19:02:58.786097Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:02:59.193181Z [debug    ] https://api.bybit.com:443 "GET /v5/account/transaction-log?accountType=UNIFIED&endTime=1785092578341&limit=50&startTime=1784487778341 HTTP/1.1" 200 107 [urllib3.connectionpool]


Fetched 0 Bybit transaction log rows


In [14]:
normalized_bybit_events = []
for payload in bybit_executions[:5]:
    try:
        normalized_bybit_events.append(normalize_bybit_spot_execution(payload))
    except Exception as exc:
        print('Could not normalize Bybit execution:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_bybit_events)} Bybit events')
if normalized_bybit_events:
    pprint(normalized_bybit_events[0])

Normalized 0 Bybit events


## OKX Direct Client Smoke Tests

These cells test signed private requests directly against OKX. They use the active OKX token stored in the app database when available, and fall back to `OKX_*` environment variables only when no DB token exists.


In [15]:
okx = get_okx_client_from_db_or_env()
if okx_db_token is not None:
    print(
        'Using DB OKX token:',
        {
            'token_id': okx_db_token.id,
            'broker': okx_db_token.broker.name,
            'account_id': okx_db_account.id if okx_db_account else None,
            'simulated_trading': okx_db_token.simulated_trading,
        },
    )
else:
    print('Using OKX_* environment credentials')

balance = okx.get_private('/api/v5/account/balance')
print('OKX balance response keys:', balance.keys())
pprint(balance.get('data', [])[:1])


Using DB OKX token: {'token_id': 1, 'broker': 'OKX', 'account_id': 18, 'simulated_trading': False}


2026-07-26T19:02:59.220608Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:02:59.698003Z [debug    ] https://www.okx.com:443 "GET /api/v5/account/balance HTTP/1.1" 200 None [urllib3.connectionpool]


OKX balance response keys: dict_keys(['code', 'data', 'msg'])
[{'adjEq': '',
  'availEq': '',
  'borrowFroz': '',
  'delta': '',
  'deltaLever': '',
  'deltaNeutralStatus': '',
  'details': [{'accAvgPx': '',
               'autoLendAmt': '0',
               'autoLendMtAmt': '0',
               'autoLendStatus': 'unsupported',
               'autoStakingStatus': 'unsupported',
               'availBal': '0.0000000018920997',
               'availEq': '0.0000000018920997',
               'borrowFroz': '',
               'cashBal': '0.0000000018920997',
               'ccy': 'BTC',
               'clSpotInUseAmt': '',
               'colBorrAutoConversion': '0',
               'colRes': '0',
               'collateralEnabled': False,
               'collateralRestrict': False,
               'crossLiab': '',
               'disEq': '0',
               'eq': '0.0000000018920997',
               'eqUsd': '0.00012235584367',
               'fixedBal': '0',
               'frozenBal': '0',
  

In [16]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
okx_fill_params = {
    'instType': 'SPOT',
    'begin': start_ms,
    'end': end_ms,
}

okx_fills = []
try:
    okx_fills = list(okx.iter_fills_history(okx_fill_params))
except CryptoExchangeAPIError as exc:
    print('OKX fills fetch failed:', exc)

print(f'Fetched {len(okx_fills)} OKX fill rows')
if okx_fills:
    pprint(okx_fills[0])

2026-07-26T19:03:00.035327Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:03:00.525260Z [debug    ] https://www.okx.com:443 "GET /api/v5/trade/fills-history?begin=1784487780031&end=1785092580031&instType=SPOT HTTP/1.1" 200 31 [urllib3.connectionpool]


Fetched 0 OKX fill rows


In [17]:
normalized_okx_events = []
for payload in okx_fills[:5]:
    try:
        normalized_okx_events.append(normalize_okx_spot_fill(payload))
    except Exception as exc:
        print('Could not normalize OKX fill:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_okx_events)} OKX events')
if normalized_okx_events:
    pprint(normalized_okx_events[0])

Normalized 0 OKX events


## List Exchange Transactions For A Period

Lists read-only raw exchange activity for a selected period. Defaults to the last `CRYPTO_API_LOOKBACK_DAYS` days. Override with:

- `CRYPTO_API_DATE_FROM=YYYY-MM-DD`
- `CRYPTO_API_DATE_TO=YYYY-MM-DD`
- `CRYPTO_API_TRANSACTION_PROVIDER=all|bybit|okx`
- `CRYPTO_API_MAX_TRANSACTION_PAGES=20`

Bybit sources:

- `/v5/execution/list` for fills
- `/v5/account/transaction-log` for account movements, Earn activity, rewards, transfers, funding-like rows

OKX sources:

- `/api/v5/trade/fills-history` for fills
- `/api/v5/account/bills-archive` and `/api/v5/account/bills` for account bills
- `/api/v5/asset/bills` for funding/asset bills where permissions allow it


In [18]:
def period_bounds():
    env_from = os.getenv('CRYPTO_API_DATE_FROM')
    env_to = os.getenv('CRYPTO_API_DATE_TO')
    if env_to:
        date_to = datetime.strptime(env_to, '%Y-%m-%d').date()
    else:
        date_to = datetime.now(timezone.utc).date()
    if env_from:
        date_from = datetime.strptime(env_from, '%Y-%m-%d').date()
    else:
        date_from = date_to - timedelta(days=LOOKBACK_DAYS)
    return date_from, date_to


def date_to_start_ms(day):
    return str(int(datetime.combine(day, datetime.min.time(), tzinfo=timezone.utc).timestamp() * 1000))


def date_to_end_ms(day):
    return str(int(datetime.combine(day, datetime.max.time(), tzinfo=timezone.utc).timestamp() * 1000))


def ms_to_utc(value):
    if value in (None, ''):
        return None
    try:
        return datetime.fromtimestamp(int(value) / 1000, tz=timezone.utc).isoformat()
    except Exception:
        return None


def compact_bybit_execution(row):
    timestamp_ms = row.get('execTime') or row.get('createdTime')
    return {
        'provider': 'bybit',
        'source': 'execution',
        'timestamp_ms': timestamp_ms,
        'timestamp_utc': ms_to_utc(timestamp_ms),
        'id': row.get('execId'),
        'order_id': row.get('orderId'),
        'symbol': row.get('symbol'),
        'side': row.get('side'),
        'quantity': row.get('execQty'),
        'price': row.get('execPrice'),
        'fee': row.get('execFee'),
        'fee_currency': row.get('feeCurrency'),
        'type': row.get('execType'),
        'raw': row,
    }


def compact_bybit_log(row):
    timestamp_ms = row.get('transactionTime')
    return {
        'provider': 'bybit',
        'source': 'transaction_log',
        'timestamp_ms': timestamp_ms,
        'timestamp_utc': ms_to_utc(timestamp_ms),
        'id': row.get('id'),
        'order_id': row.get('orderId'),
        'symbol': row.get('symbol'),
        'currency': row.get('currency'),
        'side': row.get('side'),
        'quantity': row.get('qty'),
        'cash_flow': row.get('cashFlow'),
        'change': row.get('change'),
        'fee': row.get('fee'),
        'type': row.get('type'),
        'sub_type': row.get('transSubType'),
        'raw': row,
    }


def compact_okx_fill(row):
    timestamp_ms = row.get('fillTime') or row.get('ts')
    return {
        'provider': 'okx',
        'source': 'fill',
        'timestamp_ms': timestamp_ms,
        'timestamp_utc': ms_to_utc(timestamp_ms),
        'id': row.get('tradeId') or row.get('billId'),
        'order_id': row.get('ordId'),
        'instrument': row.get('instId'),
        'side': row.get('side'),
        'quantity': row.get('fillSz'),
        'price': row.get('fillPx'),
        'fee': row.get('fee'),
        'fee_currency': row.get('feeCcy'),
        'type': row.get('execType') or row.get('subType'),
        'raw': row,
    }


def compact_okx_bill(row, source):
    timestamp_ms = row.get('ts') or row.get('uTime') or row.get('cTime')
    return {
        'provider': 'okx',
        'source': source,
        'timestamp_ms': timestamp_ms,
        'timestamp_utc': ms_to_utc(timestamp_ms),
        'id': row.get('billId'),
        'instrument': row.get('instId'),
        'currency': row.get('ccy'),
        'quantity': row.get('sz'),
        'balance_change': row.get('balChg'),
        'balance': row.get('bal'),
        'fee': row.get('fee'),
        'type': row.get('type'),
        'sub_type': row.get('subType'),
        'raw': row,
    }


def iter_okx_rows(path, params, cursor_field='billId'):
    after = None
    pages = 0
    while pages < MAX_TRANSACTION_PAGES:
        page_params = dict(params)
        if after:
            page_params['after'] = after
        data = okx_for_transactions.get_private(path, page_params)
        rows = data.get('data') or []
        for row in rows:
            yield row
        if not rows:
            break
        after = rows[-1].get(cursor_field)
        if not after:
            break
        pages += 1


def without_raw(row):
    return {key: value for key, value in row.items() if key != 'raw'}


date_from, date_to = period_bounds()
start_ms = date_to_start_ms(date_from)
end_ms = date_to_end_ms(date_to)
print('Transaction period:', {'date_from': str(date_from), 'date_to': str(date_to)})

transaction_rows = []

if TRANSACTION_PROVIDER in {'all', 'bybit'} and bybit_db_token is not None:
    bybit_for_transactions = get_bybit_client_from_db_or_env()
    bybit_category = os.getenv('BYBIT_CATEGORY', 'spot')
    bybit_log_account_type = os.getenv('BYBIT_ACCOUNT_TYPE', 'UNIFIED')
    execution_params = {'category': bybit_category, 'startTime': start_ms, 'endTime': end_ms}
    log_params = {'accountType': bybit_log_account_type, 'startTime': start_ms, 'endTime': end_ms}
    try:
        transaction_rows.extend(
            compact_bybit_execution(row) for row in bybit_for_transactions.iter_executions(execution_params)
        )
    except CryptoExchangeAPIError as exc:
        print('Bybit execution listing failed:', exc)
    try:
        transaction_rows.extend(
            compact_bybit_log(row) for row in bybit_for_transactions.iter_transaction_log(log_params)
        )
    except CryptoExchangeAPIError as exc:
        print('Bybit transaction-log listing failed:', exc)

if TRANSACTION_PROVIDER in {'all', 'okx'} and okx_db_token is not None:
    okx_for_transactions = get_okx_client_from_db_or_env()
    fill_params = {'instType': 'SPOT', 'begin': start_ms, 'end': end_ms}
    bill_params = {'begin': start_ms, 'end': end_ms}
    try:
        transaction_rows.extend(
            compact_okx_fill(row) for row in okx_for_transactions.iter_fills_history(fill_params)
        )
    except CryptoExchangeAPIError as exc:
        print('OKX fills listing failed:', exc)
    for label, path in [
        ('account_bills_archive', '/api/v5/account/bills-archive'),
        ('account_bills', '/api/v5/account/bills'),
        ('asset_bills', '/api/v5/asset/bills'),
    ]:
        try:
            transaction_rows.extend(
                compact_okx_bill(row, label) for row in iter_okx_rows(path, bill_params)
            )
        except CryptoExchangeAPIError as exc:
            print(f'OKX {label} listing failed:', exc)

transaction_rows.sort(key=lambda row: row.get('timestamp_ms') or '')
print(f'Fetched {len(transaction_rows)} exchange transaction/activity rows')
pprint([without_raw(row) for row in transaction_rows[:50]])

if transaction_rows:
    print('First raw row is available as transaction_rows[0]["raw"]')


Transaction period: {'date_from': '2026-07-19', 'date_to': '2026-07-26'}


2026-07-26T19:03:00.561561Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:03:00.969835Z [debug    ] https://api.bybit.com:443 "GET /v5/execution/list?category=spot&endTime=1785110399999&limit=100&startTime=1784419200000 HTTP/1.1" 200 144 [urllib3.connectionpool]


Bybit execution listing failed: Bybit API error: The time range between startTime and endTime cannot exceed 7 days.


2026-07-26T19:03:00.977423Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:03:01.414403Z [debug    ] https://api.bybit.com:443 "GET /v5/account/transaction-log?accountType=UNIFIED&endTime=1785110399999&limit=50&startTime=1784419200000 HTTP/1.1" 200 144 [urllib3.connectionpool]


Bybit transaction-log listing failed: Bybit API error: The time range between startTime and endTime cannot exceed 7 days.


2026-07-26T19:03:01.423536Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:03:01.995524Z [debug    ] https://www.okx.com:443 "GET /api/v5/trade/fills-history?begin=1784419200000&end=1785110399999&instType=SPOT HTTP/1.1" 200 31 [urllib3.connectionpool]
2026-07-26T19:03:02.002548Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:03:02.504920Z [debug    ] https://www.okx.com:443 "GET /api/v5/account/bills-archive?begin=1784419200000&end=1785110399999 HTTP/1.1" 200 31 [urllib3.connectionpool]
2026-07-26T19:03:02.511742Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionpool]
2026-07-26T19:03:02.995549Z [debug    ] https://www.okx.com:443 "GET /api/v5/account/bills?begin=1784419200000&end=1785110399999 HTTP/1.1" 200 31 [urllib3.connectionpool]
2026-07-26T19:03:03.001880Z [debug    ] Starting new HTTPS connection (1): www.okx.com:443 [urllib3.connectionp

Fetched 100 exchange transaction/activity rows
[{'balance': '548.84876866',
  'balance_change': '1.66411627',
  'currency': 'BABY',
  'fee': None,
  'id': '103563358377',
  'instrument': None,
  'provider': 'okx',
  'quantity': None,
  'source': 'asset_bills',
  'sub_type': None,
  'timestamp_ms': '1783400376000',
  'timestamp_utc': '2026-07-07T04:59:36+00:00',
  'type': '89'},
 {'balance': '586.99818241',
  'balance_change': '38.14941375',
  'currency': 'BABY',
  'fee': None,
  'id': '103563358546',
  'instrument': None,
  'provider': 'okx',
  'quantity': None,
  'source': 'asset_bills',
  'sub_type': None,
  'timestamp_ms': '1783400378000',
  'timestamp_utc': '2026-07-07T04:59:38+00:00',
  'type': '89'},
 {'balance': '587.11107522',
  'balance_change': '0.11289281',
  'currency': 'BABY',
  'fee': None,
  'id': '103564338751',
  'instrument': None,
  'provider': 'okx',
  'quantity': None,
  'source': 'asset_bills',
  'sub_type': None,
  'timestamp_ms': '1783404569000',
  'timestamp_ut

## Test The Project BrokerAPI Adapter Path

This is the preferred integration test when credentials have been stored through User Settings. It uses the selected `Broker` and `Account` instances discovered from the database above and exercises the same adapter path used by Direct Import.

The cell fetches normalized events only. It does not persist transactions.


In [19]:
if selected_provider == 'bybit':
    adapter = BybitAPI()
elif selected_provider == 'okx':
    adapter = OKXAPI()
else:
    raise RuntimeError('No selected provider. Run Database Credential Discovery first.')

account = selected_account
if account is None:
    raise RuntimeError('No selected account. Run Database Credential Discovery first.')

date_to = datetime.now(timezone.utc).date().isoformat()
date_from = (datetime.now(timezone.utc).date() - timedelta(days=LOOKBACK_DAYS)).isoformat()

print(
    'Fetching via BrokerAPI:',
    {
        'adapter': adapter.__class__.__name__,
        'broker': account.broker.name,
        'account_id': account.id,
        'account': account.name,
        'date_from': date_from,
        'date_to': date_to,
    },
)

await adapter.connect(user)
adapter_events = []
async for event in adapter.get_transactions(account, date_from=date_from, date_to=date_to):
    adapter_events.append(event)
await adapter.disconnect()

print(f'Fetched {len(adapter_events)} normalized events through {adapter.__class__.__name__}')
if adapter_events:
    pprint(adapter_events[0])


Fetching via BrokerAPI: {'adapter': 'BybitAPI', 'broker': 'ByBit', 'account_id': 17, 'account': 'Main', 'date_from': '2026-07-19', 'date_to': '2026-07-26'}


2026-07-26T19:03:05.084411Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:03:05.480050Z [debug    ] https://api.bybit.com:443 "GET /v5/execution/list?category=spot&endTime=1785110399999&limit=100&startTime=1784419200000 HTTP/1.1" 200 144 [urllib3.connectionpool]
2026-07-26T19:03:05.486296Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:03:05.931902Z [debug    ] https://api.bybit.com:443 "GET /v5/execution/list?category=option&endTime=1785110399999&limit=100&startTime=1784419200000 HTTP/1.1" 200 144 [urllib3.connectionpool]
2026-07-26T19:03:05.938224Z [debug    ] Starting new HTTPS connection (1): api.bybit.com:443 [urllib3.connectionpool]
2026-07-26T19:03:06.358402Z [debug    ] https://api.bybit.com:443 "GET /v5/asset/deposit/query-record?endTime=1785110399999&startTime=1784419200000 HTTP/1.1" 200 110 [urllib3.connectionpool]
2026-07-26T19:03:06.365577Z [debug    ] Startin

Fetched 0 normalized events through BybitAPI


## Optional: Persist One Normalized Event

This writes `Transactions` rows. Keep `CRYPTO_API_TEST_ALLOW_DB_WRITES=0` unless you are intentionally testing persistence in a disposable database or with a transaction you are prepared to delete.

The importer is idempotent by provider/account/event id, but this is still a database write.

In [20]:
if not ALLOW_DB_WRITES:
    raise RuntimeError(
        'Database writes are disabled. Set CRYPTO_API_TEST_ALLOW_DB_WRITES=1 to run this cell.'
    )

if adapter_events:
    events_to_persist = adapter_events
    account_to_persist = selected_account
elif normalized_bybit_events and bybit_db_account:
    events_to_persist = normalized_bybit_events
    account_to_persist = bybit_db_account
elif normalized_okx_events and okx_db_account:
    events_to_persist = normalized_okx_events
    account_to_persist = okx_db_account
else:
    raise RuntimeError('No normalized events with a matching DB account are available to persist')

created = persist_crypto_exchange_event(events_to_persist[0], user, account_to_persist)
print(f'Created {len(created)} Transactions rows')
for tx in created:
    print(tx.id, tx.type, tx.security, tx.quantity, tx.price, tx.import_provider, tx.import_event_id)


RuntimeError: Database writes are disabled. Set CRYPTO_API_TEST_ALLOW_DB_WRITES=1 to run this cell.

## Live Transaction-Type Preview (no DB writes)

Fetches **real rows** from every exchange endpoint we support — spot, option
premium, deposit, withdrawal, earn reward, option settlement, for both Bybit
and OKX — normalizes each through the real normalizer, and renders the exact
`Transactions` row(s) the import would create in production. Nothing is persisted.

This validates real response shapes (not synthetic payloads), so it surfaces
field-name or endpoint-path mismatches the moment credentials or endpoints
change. Cap rows per source with `CRYPTO_API_PREVIEW_CAP` (default 5).


In [ ]:
# Live-data transaction-type preview (no DB writes).
#
# Fetches REAL rows from every exchange endpoint we wired up in this work
# (spot, option premium, deposit, withdrawal, earn reward, option settlement
# — for both Bybit and OKX), normalizes each through the real normalizer, and
# renders the exact `Transactions` row(s) the import would create in production.
#
# This validates the real response shapes — not synthetic payloads — so it
# surfaces field-name mismatches (e.g. settlement price source, OKX
# deposit-withdraw `type` codes) the moment credentials + endpoints change.
#
# Depends on cell-2 setup, the smoke check, and the get_*_client_from_db_or_env
# helpers from cell 6. Safe to re-run any time.
from decimal import Decimal

# Cell-local imports for the quantized price preview below.
# Transactions is also imported in cell 2; re-imported here so this cell
# is self-contained if run in isolation after a kernel restart.
from common.models import Transactions
from services.crypto_exchange import _normalize_model_decimal

from services.crypto_exchange import (
    _event_datetime,
    _leg_quantity,
    _transaction_type_for_event,
    normalize_bybit_deposit,
    normalize_bybit_option_execution,
    normalize_bybit_option_settlement,
    normalize_bybit_reward,
    normalize_bybit_spot_execution,
    normalize_bybit_withdrawal,
    normalize_okx_deposit,
    normalize_okx_option_fill,
    normalize_okx_option_settlement,
    normalize_okx_reward,
    normalize_okx_spot_fill,
    normalize_okx_withdrawal,
)


def _quantized_price(leg):
    """Quantize the preview price the way persist would (9 dp), with fallback.

    Mirrors _normalize_model_decimal(Transactions, "price", ...) so the preview
    shows the exact value the DB will store. Falls back to the raw value if
    quantization overflows (preserves the "shows what is wrong" diagnostic
    when a price is genuinely out-of-range).
    """
    raw = leg.get("price")
    if raw is None:
        return None
    try:
        return str(_normalize_model_decimal(Transactions, "price", raw))
    except ValueError:
        return str(raw)


# Knobs you can edit directly in this cell (no kernel restart needed):
#   PREVIEW_DAYS  — how many days back to fetch (defaults to LOOKBACK_DAYS).
#   PREVIEW_CAP   — max rows fetched+rendered per source (raises with a busy
#                   account; note some endpoints e.g. OKX deposit-history
#                   ignore the date window and return most-recent-N anyway).
# Both still honor the CRYPTO_API_PREVIEW_CAP / CRYPTO_API_PREVIEW_DAYS env
# vars if you prefer to set them before launching Jupyter.
PREVIEW_DAYS = int(os.getenv("CRYPTO_API_PREVIEW_DAYS") or LOOKBACK_DAYS)
PREVIEW_CAP = int(os.getenv("CRYPTO_API_PREVIEW_CAP", "5"))


def _take(iterable, cap):
    count = 0
    for item in iterable:
        if count >= cap:
            break
        count += 1
        yield item


def _render_row(label, provider, index, leg, event):
    """Render one leg the way persist_crypto_exchange_event would store it."""
    quantity = _leg_quantity(leg)
    tx_type = _transaction_type_for_event(event, quantity)
    event_id = f"{event.provider_event_id}:{index}"
    when = _event_datetime(event).isoformat()
    return {
        "scenario": label,
        "import_provider": provider,
        "import_event_id": event_id,
        "import_group_id": event.group_id,
        "import_event_type": event.category,
        "type (Transactions.type)": tx_type,
        "security (resolved asset)": f"{leg['asset']}  [instrument={leg.get('instrument', 'coin')}]",
        "quantity (DecimalField 25,9)": str(quantity),
        "price (DecimalField 18,9)": _quantized_price(leg),
        "price_asset": leg.get("price_asset"),
        "currency": "USD",
        "date (naive UTC)": when,
    }


# Date window reused across iterators (matches the rest of the notebook).
_preview_start, _preview_end = date_range_ms(PREVIEW_DAYS)
bybit_date_params = {"startTime": _preview_start, "endTime": _preview_end}
okx_date_params = {"begin": _preview_start, "end": _preview_end}

# (scenario label, provider, normalizer, client, method_name, params)
# Mirrors the (iterator -> normalizer) pairings in BybitAPI/OKXAPI.get_transactions.
SOURCES = []
if bybit_db_token is not None:
    bybit_client = get_bybit_client_from_db_or_env()
    SOURCES += [
        ("Bybit spot trade", "bybit", normalize_bybit_spot_execution, bybit_client, "iter_executions", {"category": "spot", **bybit_date_params}),
        ("Bybit option premium", "bybit", normalize_bybit_option_execution, bybit_client, "iter_option_executions", bybit_date_params),
        ("Bybit deposit", "bybit", normalize_bybit_deposit, bybit_client, "iter_deposits", bybit_date_params),
        ("Bybit withdrawal", "bybit", normalize_bybit_withdrawal, bybit_client, "iter_withdrawals", bybit_date_params),
        ("Bybit earn reward", "bybit", normalize_bybit_reward, bybit_client, "iter_transaction_log", {"type": "Earn", **bybit_date_params}),
        ("Bybit option settlement", "bybit", normalize_bybit_option_settlement, bybit_client, "iter_option_settlements", bybit_date_params),
    ]
if okx_db_token is not None:
    okx_client = get_okx_client_from_db_or_env()
    SOURCES += [
        ("OKX spot fill", "okx", normalize_okx_spot_fill, okx_client, "iter_fills_history", {"instType": "SPOT", **okx_date_params}),
        ("OKX option fill", "okx", normalize_okx_option_fill, okx_client, "iter_option_fills", {"instType": "OPTION", **okx_date_params}),
        ("OKX deposit", "okx", normalize_okx_deposit, okx_client, "iter_deposits", okx_date_params),
        ("OKX withdrawal", "okx", normalize_okx_withdrawal, okx_client, "iter_withdrawals", okx_date_params),
        ("OKX earn reward", "okx", normalize_okx_reward, okx_client, "iter_earn_lending_history", okx_date_params),
        ("OKX option settlement", "okx", normalize_okx_option_settlement, okx_client, "iter_option_settlements", okx_date_params),
    ]

if not SOURCES:
    raise RuntimeError(
        "No stored Bybit/OKX tokens for this user. Save credentials via User "
        "Settings and run Database Credential Discovery (cell 6) first."
    )

print(f"Live fetch over {PREVIEW_DAYS} day(s) (LOOKBACK_DAYS={LOOKBACK_DAYS}), capped at {PREVIEW_CAP} row(s) per source.")
print(f"Sources: {len(SOURCES)} iterator/normalizer pairs across "
      f"{sum(1 for s in SOURCES if s[1]=='bybit')} Bybit + {sum(1 for s in SOURCES if s[1]=='okx')} OKX.")
print("Nothing is written to the database.\n")

rendered_rows = []
per_source_counts = []

for label, provider, normalizer, client, method_name, params in SOURCES:
    print("=" * 78)
    print(f"SOURCE: {label}  ({method_name})")
    iterator = getattr(client, method_name)
    fetched = 0
    normalized = 0
    failed = 0
    try:
        for payload in _take(iterator(params), PREVIEW_CAP):
            fetched += 1
            try:
                event = normalizer(payload)
            except Exception as exc:
                failed += 1
                print(f"  normalize failed on payload: {exc!r}")
                print(f"    payload keys: {list(payload.keys())}")
                continue
            if event is None:
                # Normalizer deliberately suppressed this row (e.g. internal-transfer filter).
                continue
            normalized += 1
            print(f"  payload (raw): {payload}")
            print(f"  -> CryptoExchangeEvent(category={event.category!r}, raw_type={event.raw_type!r}, "
                  f"provider_event_id={event.provider_event_id!r}, {len(event.legs)} leg(s))")
            for idx, leg in enumerate(event.legs):
                if leg.get("role") == "fee":
                    print(f"     leg {idx}: FEE-ONLY (skipped by persist; asset={leg.get('asset')})")
                    continue
                row = _render_row(label, provider, idx, leg, event)
                rendered_rows.append(row)
                print(f"     leg {idx} -> Transactions row:")
                for key, value in row.items():
                    print(f"        {key}: {value}")
            print()
    except Exception as exc:
        # Network/permission errors per source — record and continue so one
        # failing endpoint doesn't hide the rest (same isolation policy as the adapter).
        print(f"  FETCH FAILED: {exc!r}")
    if fetched == 0:
        print("  (no rows in window — try raising CRYPTO_API_LOOKBACK_DAYS)")
    per_source_counts.append((label, fetched, normalized, failed))
    print()

print("=" * 78)
print("PER-SOURCE SUMMARY  (fetched / normalized / failed)")
for label, fetched, normalized, failed in per_source_counts:
    print(f"  {label:32} {fetched:4} {normalized:4} {failed:4}")
print(f"\nTOTAL: {len(rendered_rows)} Transactions row(s) would be created across all sources.")

if rendered_rows:
    print("\nSchema alignment check:")
    required_keys = {
        "import_provider", "import_event_id",
        "type (Transactions.type)", "security (resolved asset)",
        "quantity (DecimalField 25,9)", "price (DecimalField 18,9)",
        "currency", "date (naive UTC)",
    }
    for r in rendered_rows:
        missing = required_keys - set(r)
        assert not missing, f"row missing keys {missing}: {r}"
    print(f"  all {len(rendered_rows)} rendered rows carry the required Transactions columns.")
    print("\nTX TYPE DISTRIBUTION:")
    from collections import Counter
    for tx_type, count in Counter(r["type (Transactions.type)"] for r in rendered_rows).most_common():
        print(f"  {count:4}  {tx_type}")
else:
    print("\nNo rows rendered — see per-source output above. If all sources fetched 0,")
    print("raise CRYPTO_API_LOOKBACK_DAYS. If sources FAILED, check the error messages")
    print("(often API-key permission scope, e.g. asset-history or options read missing).")


## Troubleshooting Notes

- `No usable stored exchange token/account was found`: confirm migrations are applied, credentials are saved through User Settings, and a broker account exists under the same broker.
- `no such column: common_transactions.import_provider`: run `python manage.py migrate` from the backend directory.
- Bybit `permission denied` or similar: check that the key has read access for account, wallet, transaction log, and execution history.
- Bybit testnet failures with mainnet keys: testnet and mainnet keys are not interchangeable.
- OKX `Invalid Sign`: verify the API secret and passphrase exactly. Passphrases are user-defined and case-sensitive.
- OKX simulated failures: use `simulated_trading=True` only with demo/simulated credentials.
- Empty results are valid if there were no fills/log entries in the selected date range. Increase `CRYPTO_API_LOOKBACK_DAYS`.
- Keep raw payloads when normalization fails; those payloads are the best fixtures for extending the normalizers.
